In [11]:
# ---------------------------
#  Imports
# ---------------------------
import numpy as np
import pandas as pd

# ---------------------------
#  Month mapping
# ---------------------------
def build_month_codes():
    return {
        'Jan': 1,
        'Feb': 2,
        'Mar': 3,
        'Apr': 4,
        'May': 5,
        'Jun': 6,
        'Jul': 7,
        'Aug': 8,
        'Sep': 9,
        'Oct': 10,
        'Nov': 11,
        'Dec': 12
    }

# ---------------------------
#  Parse id into month text and sector string
# ---------------------------
def split_test_id_column(df):
    parts = df.id.str.split('_', expand=True)
    df['month_text'] = parts[0]
    df['sector'] = parts[1]
    return df

# ---------------------------
#  Add parsed time fields to a dataframe
# ---------------------------
def add_time_and_sector_fields(df, month_codes):
    if 'sector' in df.columns:
        df['sector_id'] = df.sector.str.slice(7, None).astype(int)
    if 'month' not in df.columns:
        df['month'] = df['month_text'].str.slice(5, None).map(month_codes)
        df['year'] = df['month_text'].str.slice(0, 4).astype(int)
        df['time'] = (df['year'] - 2019) * 12 + df['month'] - 1
    else:
        df['year'] = df.month.str.slice(0, 4).astype(int)
        df['month'] = df.month.str.slice(5, None).map(month_codes)
        df['time'] = (df['year'] - 2019) * 12 + df['month'] - 1
    return df

# ---------------------------
#  Load competition tables used for submission
# ---------------------------
def load_competition_data():
    train_nht = pd.read_csv('c:/Users/arnov/Desktop/Kaggle/China/china-real-estate-demand-prediction/train/new_house_transactions.csv')
    test = pd.read_csv('c:/Users/arnov/Desktop/Kaggle/China/china-real-estate-demand-prediction/test.csv')
    return train_nht, test

# ---------------------------
#  Build training matrix: amount_new_house_transactions [time x sector_id]
# ---------------------------
def build_amount_matrix(train_nht, month_codes):
    train_nht = add_time_and_sector_fields(train_nht.copy(), month_codes)
    pivot = train_nht.set_index(['time', 'sector_id']).amount_new_house_transactions.unstack()
    pivot = pivot.fillna(0)
    all_sectors = np.arange(1, 97)
    for s in all_sectors:
        if s not in pivot.columns:
            pivot[s] = 0
    pivot = pivot[all_sectors]
    return pivot

# ---------------------------
#  Compute sector-level December multipliers from training
# ---------------------------
def compute_december_multipliers(a_tr, eps=1e-9, min_dec_obs=1, clip_low=0.8, clip_high=1.5):
    is_december = (a_tr.index.values % 12) == 11
    dec_means = a_tr[is_december].mean(axis=0)
    nondec_means = a_tr[~is_december].mean(axis=0)
    dec_counts = a_tr[is_december].notna().sum(axis=0)
    raw_mult = dec_means / (nondec_means + eps)
    overall_mult = float(dec_means.mean() / (nondec_means.mean() + eps))
    raw_mult = raw_mult.where(dec_counts >= min_dec_obs, overall_mult)
    raw_mult = raw_mult.replace([np.inf, -np.inf], 1.0).fillna(1.0)
    clipped_mult = raw_mult.clip(lower=clip_low, upper=clip_high)
    return clipped_mult.to_dict()

# ---------------------------
#  Apply December bump on the forecast horizon
# ---------------------------
def apply_december_bump(a_pred, sector_to_mult):
    dec_rows = [t for t in a_pred.index.values if (t % 12) == 11]
    if len(dec_rows) == 0:
        return a_pred
    for sector in a_pred.columns:
        m = sector_to_mult.get(sector, 1.0)
        a_pred.loc[dec_rows, sector] = a_pred.loc[dec_rows, sector] * m
    return a_pred

# ---------------------------
#  Exponential weighted geometric mean per sector
# ---------------------------
def ewgm_per_sector(a_tr, sector, n_lags, alpha):
    weights = np.array([alpha**(n_lags - 1 - i) for i in range(n_lags)], dtype=float)
    weights = weights / weights.sum()
    recent_vals = a_tr.tail(n_lags)[sector].values
    if (len(recent_vals) != n_lags) or (recent_vals <= 0).all():
        return 0.0
    mask = recent_vals > 0
    pos_vals = recent_vals[mask]
    pos_w = weights[mask]
    if pos_vals.size == 0:
        return 0.0
    pos_w = pos_w / pos_w.sum()
    log_vals = np.log(pos_vals + 1e-12)
    wlm = np.sum(pos_w * log_vals) / pos_w.sum()
    return float(np.exp(wlm))

# ---------------------------
#  Build horizon predictions [time=67..78 x sectors]
# ---------------------------
def predict_horizon(a_tr, n_lags, alpha, t2):
    idx = np.arange(67, 79)
    cols = a_tr.columns
    a_pred = pd.DataFrame(index=idx, columns=cols, dtype=float)
    for sector in cols:
        if (a_tr.tail(t2)[sector].min() == 0) or (a_tr[sector].sum() == 0):
            a_pred[sector] = 0.0
            continue
        base = ewgm_per_sector(a_tr=a_tr, sector=sector, n_lags=n_lags, alpha=alpha)
        a_pred[sector] = base
    a_pred.index.rename('time', inplace=True)
    return a_pred

# ---------------------------
#  Convert wide predictions into submission aligned with test ids
# ---------------------------
def build_submission_df(a_pred, test_raw, month_codes):
    test = split_test_id_column(test_raw.copy())
    test = add_time_and_sector_fields(test, month_codes)
    lookup = a_pred.stack().rename('pred').reset_index().rename(columns={'level_1': 'sector_id'})
    merged = test.merge(lookup, how='left', on=['time', 'sector_id'])
    merged['pred'] = merged['pred'].fillna(0.0)
    out = merged[['id', 'pred']].rename(columns={'pred': 'new_house_transaction_amount'})
    return out

# ---------------------------
#  End-to-end generation with December bump
# ---------------------------
def generate_submission_with_december_bump(n_lags=6, alpha=0.5, t2=6, clip_low=0.85, clip_high=1.4):
    month_codes = build_month_codes()
    train_nht, test = load_competition_data()
    a_tr = build_amount_matrix(train_nht, month_codes)
    a_pred = predict_horizon(a_tr=a_tr, n_lags=n_lags, alpha=alpha, t2=t2)
    sector_to_mult = compute_december_multipliers(a_tr=a_tr, eps=1e-9, min_dec_obs=1, clip_low=clip_low, clip_high=clip_high)
    a_pred = apply_december_bump(a_pred=a_pred, sector_to_mult=sector_to_mult)
    submission = build_submission_df(a_pred=a_pred, test_raw=test, month_codes=month_codes)
    submission.to_csv('submission.csv', index=False)
    return a_tr, a_pred, submission

In [12]:
# Weight Decay Geometric Mean method (modular, no file output)
def ewgm_weight_decay(a_tr, sector, n_lags=6, alpha=0.5, decay=0.98):
    weights = np.array([alpha**(n_lags - 1 - i) for i in range(n_lags)], dtype=float)
    weights = weights * (decay ** np.arange(n_lags)[::-1])
    weights = weights / weights.sum()
    recent_vals = a_tr.tail(n_lags)[sector].values
    if (len(recent_vals) != n_lags) or (recent_vals <= 0).all():
        return 0.0
    mask = recent_vals > 0
    pos_vals = recent_vals[mask]
    pos_w = weights[mask]
    if pos_vals.size == 0:
        return 0.0
    pos_w = pos_w / pos_w.sum()
    log_vals = np.log(pos_vals + 1e-12)
    wlm = np.sum(pos_w * log_vals) / pos_w.sum()
    return float(np.exp(wlm))

def predict_horizon_weight_decay(a_tr, n_lags=6, alpha=0.5, t2=6, decay=0.98):
    idx = np.arange(67, 79)
    cols = a_tr.columns
    a_pred = pd.DataFrame(index=idx, columns=cols, dtype=float)
    for sector in cols:
        if (a_tr.tail(t2)[sector].min() == 0) or (a_tr[sector].sum() == 0):
            a_pred[sector] = 0.0
            continue
        base = ewgm_weight_decay(a_tr=a_tr, sector=sector, n_lags=n_lags, alpha=alpha, decay=decay)
        a_pred[sector] = base
    a_pred.index.rename('time', inplace=True)
    return a_pred

In [13]:
# Pipeline: generate bump and weight decay predictions, ensemble, and save only the ensemble submission
import pandas as pd

# Generate bump predictions
a_tr_bump, a_pred_bump, sub_bump = generate_submission_with_december_bump(n_lags=7, alpha=0.5, t2=6, clip_low=0.85, clip_high=1.4)

# Extract test and month_codes for weight decay
month_codes = build_month_codes()
train_nht, test = load_competition_data()

# Generate weight decay predictions
a_pred_wd = predict_horizon_weight_decay(a_tr=a_tr_bump, n_lags=7, alpha=0.5, t2=6, decay=0.98)
sub_wd = build_submission_df(a_pred=a_pred_wd, test_raw=test, month_codes=month_codes)

# Ensemble: weighted average
weight_bump = 0.35
weight_wd = 0.55
ensemble = sub_bump.copy()
ensemble['new_house_transaction_amount'] = (
    weight_bump * sub_bump['new_house_transaction_amount'] +
    weight_wd * sub_wd['new_house_transaction_amount']
)
ensemble.to_csv('submission_ensemble.csv', index=False)
print(' submission_ensemble.csv file created successfully')

 submission_ensemble.csv file created successfully


In [2]:
# -*- coding: utf-8 -*-
# Ensemble EWGM + EWGM(weight-decay) + Monthly Seasonality Bump
# - Index temps absolu (year*12 + month)
# - Apprentissage du poids d'ensemble par validation roulante (SMAPE)
# - Bump saisonnier par mois & secteur, normalisé à 1
# - Zéro-safe, sans indices temporels codés en dur

import numpy as np
import pandas as pd

# ---------------------------
# Config chemins (à adapter)
# ---------------------------
TRAIN_PATH = r"c:/Users/arnov/Desktop/Kaggle/China/china-real-estate-demand-prediction/train/new_house_transactions.csv"
TEST_PATH  = r"c:/Users/arnov/Desktop/Kaggle/China/china-real-estate-demand-prediction/test.csv"
OUT_PATH   = r"submission_ensemble.csv"

# ---------------------------
# Utils mois / parsing
# ---------------------------
MONTH_MAP = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,
             "Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12}

def add_time_and_sector_fields(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "month" in df.columns:
        df["year"]  = df["month"].str[:4].astype(int)
        df["mstr"]  = df["month"].str[5:]
    else:
        # test has month_text inside id
        df["year"]  = df["month_text"].str[:4].astype(int)
        df["mstr"]  = df["month_text"].str[5:]
    df["month_num"] = df["mstr"].map(MONTH_MAP).astype(int)
    df["time"] = df["year"]*12 + df["month_num"]
    # month_id 0..11
    df["month_id"] = (df["time"] - 1) % 12
    # sector_id
    if "sector" in df.columns:
        df["sector_id"] = df["sector"].astype(str).str.extract(r"(\d+)$").iloc[:,0].astype(int)
    elif "sector_id" in df.columns:
        df["sector_id"] = df["sector_id"].astype(str).str.extract(r"(\d+)$").iloc[:,0].astype(int)
    return df

def load_competition_data():
    train = pd.read_csv(TRAIN_PATH)
    test  = pd.read_csv(TEST_PATH)
    # parse test ids
    parts = test["id"].str.split("_", expand=True)
    test["month_text"] = parts[0]
    test["sector"]     = parts[1]
    return train, test

# ---------------------------
# Matrice train [time x sector]
# ---------------------------
def build_amount_matrix(train_nht: pd.DataFrame) -> pd.DataFrame:
    df = add_time_and_sector_fields(train_nht)
    # colonne cible (robuste aux noms légèrement différents)
    target = next((c for c in df.columns if "amount" in c.lower() and "new" in c.lower() and "house" in c.lower()), None)
    if target is None:
        raise KeyError("Colonne cible (new house amount) introuvable dans le train")
    pivot = df.set_index(["time","sector_id"])[target].unstack()
    # fill (on traite l'absence comme 0 sur ce concours)
    pivot = pivot.fillna(0.0)
    pivot.index.name = "time"
    # trie colonnes et index
    pivot = pivot.sort_index().sort_index(axis=1)
    return pivot

# ---------------------------
# Horizon test (times ordonnés)
# ---------------------------
def infer_test_times(test_df: pd.DataFrame) -> np.ndarray:
    tdf = add_time_and_sector_fields(test_df)
    return np.sort(tdf["time"].unique())

# ---------------------------
# SMAPE
# ---------------------------
def smape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)).clip(1e-9, None)
    return float(np.mean(np.abs(y_pred - y_true) / denom))

# ---------------------------
# EWGM (géométrique) variantes
# ---------------------------
def _weighted_geomean(values: np.ndarray, weights: np.ndarray, eps=1e-12) -> float:
    # garde uniquement >0
    mask = values > 0
    if not np.any(mask):
        return 0.0
    v = values[mask]
    w = weights[mask]
    w = w / (w.sum() + eps)
    return float(np.exp(np.sum(w * np.log(v + eps))))

def ewgm_last_n(y: pd.Series, n_lags=6, alpha=0.5) -> float:
    # poids exponentiels récents (sans decay additionnel)
    vals = y.tail(n_lags).to_numpy(dtype=float)
    if vals.size < n_lags or (vals > 0).sum() == 0:
        return 0.0
    w = np.array([alpha**(n_lags - 1 - i) for i in range(n_lags)], dtype=float)
    return _weighted_geomean(vals, w)

def ewgm_last_n_decay(y: pd.Series, n_lags=6, alpha=0.5, decay=0.98) -> float:
    vals = y.tail(n_lags).to_numpy(dtype=float)
    if vals.size < n_lags or (vals > 0).sum() == 0:
        return 0.0
    base_w = np.array([alpha**(n_lags - 1 - i) for i in range(n_lags)], dtype=float)
    # weight-decay additionnel vers le passé
    dec_w  = decay ** np.arange(n_lags)[::-1]
    w = base_w * dec_w
    return _weighted_geomean(vals, w)

# ---------------------------
# Validation roulante pour apprendre le poids d'ensemble
# ---------------------------
def rolling_ewgm_pred(series: pd.Series, times: np.ndarray, n_lags=6, alpha=0.5, decay=0.98):
    """
    Pour chaque t in times (t ⊂ index de series), calcule la prédiction en utilisant
    les n_lags valeurs STRICTEMENT avant t. Retourne deux vecteurs:
    - p1: EWGM standard
    - p2: EWGM avec weight-decay
    """
    p1, p2 = [], []
    for t in times:
        prev = series[series.index < t]
        p1.append(ewgm_last_n(prev, n_lags=n_lags, alpha=alpha))
        p2.append(ewgm_last_n_decay(prev, n_lags=n_lags, alpha=alpha, decay=decay))
    return np.array(p1, float), np.array(p2, float)

def learn_blend_weight(series: pd.Series, n_lags=6, alpha=0.5, decay=0.98, val_len=6, grid_step=0.05):
    """
    Apprend w ∈ [0,1] pour minimiser SMAPE sur les val_len derniers mois du train.
    pred = w*p1 + (1-w)*p2
    """
    idx = series.index.to_numpy()
    if len(idx) <= n_lags + max(3, val_len):
        return 0.5  # fallback
    val_times = idx[-val_len:]
    y_val = series.loc[val_times].to_numpy(dtype=float)
    p1, p2 = rolling_ewgm_pred(series, val_times, n_lags=n_lags, alpha=alpha, decay=decay)
    # si aucune préd >0, fallback
    if (p1<=0).all() and (p2<=0).all():
        return 0.5
    best_w, best_loss = 0.5, np.inf
    grid = np.arange(0.0, 1.0 + 1e-9, grid_step)
    for w in grid:
        pred = w*p1 + (1.0 - w)*p2
        loss = smape(y_val, pred)
        if loss < best_loss:
            best_loss, best_w = loss, w
    return float(best_w)

# ---------------------------
# Prédiction sur l'horizon (avant bump)
# ---------------------------
def predict_horizon_geo(a_tr: pd.DataFrame, test_times: np.ndarray,
                        n_lags=6, alpha=0.5, decay=0.98, val_len=6) -> pd.DataFrame:
    """
    Calcule pour chaque secteur:
    - w appris sur une mini-validation roulante récente
    - base1 = EWGM(last n)
    - base2 = EWGM_decay(last n)
    - prédiction = (w*base1 + (1-w)*base2) répliquée sur l'horizon
    """
    cols = a_tr.columns
    out  = pd.DataFrame(index=test_times, columns=cols, dtype=float)
    for s in cols:
        y = a_tr[s].astype(float)
        if y.sum() == 0 or (y.tail(n_lags) > 0).sum() == 0:
            out[s] = 0.0
            continue
        w = learn_blend_weight(y, n_lags=n_lags, alpha=alpha, decay=decay, val_len=val_len, grid_step=0.05)
        b1 = ewgm_last_n(y, n_lags=n_lags, alpha=alpha)
        b2 = ewgm_last_n_decay(y, n_lags=n_lags, alpha=alpha, decay=decay)
        base = max(0.0, w*b1 + (1.0 - w)*b2)
        out[s] = base  # constant across horizon; the bump will shape by month
    out.index.name = "time"
    return out

# ---------------------------
# Seasonality bump (12 mois)
# ---------------------------
def compute_monthly_bump(a_tr: pd.DataFrame, window=36, clip_low=0.7, clip_high=1.5) -> pd.DataFrame:
    """
    Calcule pour chaque secteur s et chaque month_id m (0..11) un facteur:
       bump[s,m] = median_recent(s,m) / mean_over_12(s)
    puis normalise pour que la moyenne sur 12 mois = 1.
    """
    df = a_tr.copy()
    df = df.sort_index()
    # month_id from index
    tmp = df.copy()
    tmp["time"] = tmp.index
    tmp["month_id"] = (tmp["time"] - 1) % 12
    # fenêtre récente
    last_t = tmp["time"].max()
    win_lo = max(tmp["time"].min(), last_t - window + 1)
    long = tmp[tmp["time"] >= win_lo].melt(id_vars=["time","month_id"], var_name="sector_id", value_name="y")
    # médiane par (secteur, mois)
    med = long.groupby(["sector_id","month_id"])["y"].median().unstack(fill_value=0.0)
    # normalisation: moyenne sur 12 mois = 1
    row_mean = med.replace(0.0, np.nan).mean(axis=1)
    bump = med.div(row_mean, axis=0).fillna(1.0)
    bump = bump.clip(lower=clip_low, upper=clip_high)
    # assure types corrects
    bump.index = bump.index.astype(int)
    bump.columns = bump.columns.astype(int)  # 0..11
    return bump  # shape [sector_id, month_id]

def apply_monthly_bump(pred_wide: pd.DataFrame, bump: pd.DataFrame, alpha=0.35) -> pd.DataFrame:
    """
    pred[t,s] *= (1 - alpha) + alpha * bump[s, month_id(t)]
    """
    out = pred_wide.copy()
    month_id = (out.index.to_series() - 1) % 12
    for t, mid in zip(out.index, month_id):
        if int(mid) in bump.columns:
            # récupère bump pour toutes les colonnes dans l'ordre
            mult = ((1.0 - alpha) + alpha * bump.loc[out.columns, int(mid)].reindex(out.columns).fillna(1.0).values)
            out.loc[t] = out.loc[t].to_numpy(float) * mult
    return out

# ---------------------------
# Soumission
# ---------------------------
def build_submission(pred_wide: pd.DataFrame, test_df: pd.DataFrame) -> pd.DataFrame:
    tdf = add_time_and_sector_fields(test_df.copy())
    lookup = (pred_wide.stack()
              .rename("pred")
              .reset_index()
              .rename(columns={"level_1":"sector_id"}))
    merged = tdf.merge(lookup, how="left", on=["time","sector_id"])
    merged["pred"] = merged["pred"].fillna(0.0)
    return merged[["id","pred"]].rename(columns={"pred":"new_house_transaction_amount"})

# ---------------------------
# Pipeline principal
# ---------------------------
def run_pipeline_ewgm_bump(n_lags=7, alpha=0.5, decay=0.98, val_len=6,
                           bump_window=36, bump_alpha=0.35,
                           clip_low=0.0, clip_high=np.inf):
    train, test = load_competition_data()
    a_tr = build_amount_matrix(train)
    test_times = infer_test_times(test)

    # base: ensemble géométrique (EWGM & EWGM_decay) avec poids appris
    pred_base = predict_horizon_geo(a_tr, test_times, n_lags=n_lags, alpha=alpha, decay=decay, val_len=val_len)

    # bump 12 mois normalisé
    bump = compute_monthly_bump(a_tr, window=min(bump_window, len(a_tr)))
    pred_bumped = apply_monthly_bump(pred_base, bump, alpha=bump_alpha)

    # clipping optionnel (par sécurité globale, ici large)
    if clip_low > 0.0 or np.isfinite(clip_high):
        pred_bumped = pred_bumped.clip(lower=clip_low, upper=clip_high)

    sub = build_submission(pred_bumped, test)
    return a_tr, pred_bumped, sub

# ---------------------------
# Exécution
# ---------------------------
if __name__ == "__main__":
    a_tr, a_pred, sub = run_pipeline_ewgm_bump(
        n_lags=7, alpha=0.5, decay=0.98, val_len=6,
        bump_window=36, bump_alpha=0.35,
        clip_low=0.0, clip_high=np.inf
    )
    sub.to_csv(OUT_PATH, index=False)
    print(f"Soumission écrite: {OUT_PATH}")


Soumission écrite: submission_ensemble.csv
